In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# for text preprocessing
import re
import spacy
import pandas as pd

from nltk.corpus import stopwords 
from nltk.stem.wordnet import WordNetLemmatizer
import string

# import numpy for matrix operation
import numpy as np

# Importing Gensim
import gensim
from gensim import corpora

In [3]:
# to suppress warnings
from warnings import filterwarnings
filterwarnings('ignore')

In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
df=pd.read_csv("/kaggle/input/text-mining2/books_new (1).csv")
df

,Title,categories,description,User_id,review/score,review/text
0,"Word Biblical Commentary Vol. 5, Numbers (budd...",['Books'],NaN,A3LOZIGHG4LB3X,1.0,Word Biblical Commentary: Numbers is a worst c...
1,Castle Barebane : A Novel Of Suspense,['Books'],NaN,A89TL80GBGNUK,3.0,"""Castle Barebane"" was another experiment by Jo..."
2,Castle Barebane : A Novel Of Suspense,['Books'],NaN,A3IRBCCB3I5PPD,4.0,"Joan Aiken, daughter of poet Conrad Aiken, was..."
3,Mademoiselle liberte: Roman (French Edition),['Books'],NaN,A3CM5H0P9LNO58,5.0,"Looking for a novel in Nice last week, I came ..."
4,Mademoiselle liberte: Roman (French Edition),['Books'],NaN,A1CU6NTKULREFJ,3.0,Lorsque le feu de la passion devient incontrla...
...,...,...,...,...,...,...
568,ngeles fugaces (Falling Angels) (Spanish Edition),['Books'],NaN,A1L43KWWR05PCS,4.0,"This is the Spanish text edition of the book, ..."
569,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A29DTN3W0WUBWT,3.0,When it was published in 1980 the Collazo was ...
570,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A1DA8ESF6OXRS8,5.0,"We, technical translators, urgently need a ren..."
571,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A3G94Q5H7LWUXD,5.0,"One of the best, if not the best technical dic..."


In [ ]:
!pip install spacy
!pip install spacy-langdetect

In [ ]:
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# To ensure consistent results
DetectorFactory.seed = 0
# Function to detect language
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return None

# Apply the function to the "review/text" column
df['language'] = df['review/text'].apply(detect_language)
df_en = df[df['language'] == 'en']

# Drop the 'language' column as it's no longer needed
df_en = df_en.drop(columns=['language'])

# Display English reviews
df_en.reset_index(drop=True, inplace=True)
df_en

In [ ]:
df_most_review = df[df['Title'].str.contains(r'\bThe Death of Outrage: Bill Clinton\b', regex=True)]

# Select the relevant columns
df_most_review = df_most_review[['Title','categories','review/score', 'review/text']]

# Display the filtered DataFrame
df_most_review
#fl=df_most_review.iloc[0]
#fl.to_dict()

# TEXT PROCESSING

In [ ]:
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from stop_words import get_stop_words
from nltk.stem.wordnet import WordNetLemmatizer

In [ ]:
from nltk.corpus import stopwords
stop_words=stopwords.words('english')
pattern = r'\b[^\d\W]+\b'
tokenizer = RegexpTokenizer(pattern)
lemmatizer = WordNetLemmatizer()

In [ ]:
remove_words = ['bennett','clinton','quot','bill', 'book','read',
                'one','mr','well','like','many','ok']
#remove_words=[]

In [ ]:
# list for tokenized documents in loop
from nltk.corpus import wordnet
texts = []

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
# loop through document list
for i in df_most_review['review/text'].items():
    # clean and tokenize document string
    raw = str(i[1]).lower()
    tokens = tokenizer.tokenize(raw)

    # remove stop words from tokens
    stopped_tokens = [raw for raw in tokens if not raw in stop_words]
    
    # remove stop words from tokens
    stopped_tokens_new = [raw for raw in stopped_tokens if not raw in remove_words]
    
    # POS tagging
    pos_tags = nltk.pos_tag(stopped_tokens_new)
    
    # Lemmatize tokens with POS tags
    lemma_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in pos_tags]
    
    # lemmatize tokens
    #lemma_tokens = [lemmatizer.lemmatize(tokens) for tokens in stopped_tokens_new]
    
    # remove word containing only single char
    new_lemma_tokens = [raw for raw in lemma_tokens if not len(raw) == 1]
    
    # add tokens to list
    texts.append(new_lemma_tokens)

# sample data
print(texts[0])

In [ ]:
from itertools import chain
from nltk.probability import FreqDist
from tabulate import tabulate

# Assuming texts is a list of lists
flattened_texts = list(chain.from_iterable(texts))

# Now create FreqDist
fdist = FreqDist(flattened_texts)
most_common_lemmas = fdist.most_common(10)
data_with_pos = []
for word in most_common_lemmas:
    # Append the word, count, and POS tag to the data_with_pos list
    data_with_pos.append(word)

# Display the data with POS tags in tabular format
print(tabulate(data_with_pos, headers=['Word', 'Frequency'], tablefmt='grid'))

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Join tokens into a single string
text_2 = ' '.join(flattened_texts)
# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_2)

# Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [16]:
dict_ = corpora.Dictionary(texts)
print(dict_)

Dictionary<3350 unique tokens: ['affect', 'assault', 'attack', 'binder', 'blackmail']...>


In [17]:
# Converting list of documents (corpus) into Document Term Matrix using the dictionary 
doc_term_matrix = [dict_.doc2bow(i) for i in texts]
doc_term_matrix[0]

[(0, 1),
 (1, 1),
 (2, 1),
 (3, 1),
 (4, 1),
 (5, 1),
 (6, 1),
 (7, 1),
 (8, 1),
 (9, 2),
 (10, 1),
 (11, 1),
 (12, 1),
 (13, 1),
 (14, 1),
 (15, 1),
 (16, 1),
 (17, 1),
 (18, 1),
 (19, 1),
 (20, 1),
 (21, 1),
 (22, 1),
 (23, 1),
 (24, 1),
 (25, 1),
 (26, 1),
 (27, 1),
 (28, 1),
 (29, 2),
 (30, 1),
 (31, 1),
 (32, 1),
 (33, 1),
 (34, 1),
 (35, 2),
 (36, 1),
 (37, 1),
 (38, 1),
 (39, 1),
 (40, 1),
 (41, 1),
 (42, 1),
 (43, 1),
 (44, 1),
 (45, 1),
 (46, 1),
 (47, 1),
 (48, 1),
 (49, 1),
 (50, 1),
 (51, 1),
 (52, 1),
 (53, 2),
 (54, 1),
 (55, 1),
 (56, 2),
 (57, 1),
 (58, 1),
 (59, 1),
 (60, 1),
 (61, 1)]

In [25]:
# Creating the object for LDA model using gensim library

Lda = gensim.models.ldamodel.LdaModel

In [26]:
# Running and Training LDA model on the document term matrix.

ldamodel = Lda(doc_term_matrix, num_topics=5, id2word = dict_, passes=1, random_state=0, eval_every=None)

In [27]:
# Prints the topics with the indexes: 0,1,2 :

ldamodel.print_topics()

# we need to manually check whethere the topics are different from one another or not

[(0,
  '0.008*"president" + 0.006*"moral" + 0.006*"american" + 0.006*"people" + 0.006*"outrage" + 0.005*"argument" + 0.004*"make" + 0.004*"country" + 0.004*"morality" + 0.003*"political"'),
 (1,
  '0.014*"outrage" + 0.010*"president" + 0.005*"american" + 0.005*"make" + 0.005*"lie" + 0.004*"right" + 0.004*"public" + 0.004*"moral" + 0.004*"people" + 0.004*"get"'),
 (2,
  '0.010*"moral" + 0.006*"president" + 0.006*"outrage" + 0.005*"make" + 0.005*"american" + 0.005*"know" + 0.005*"say" + 0.005*"argument" + 0.004*"right" + 0.004*"morality"'),
 (3,
  '0.006*"president" + 0.005*"american" + 0.005*"outrage" + 0.004*"moral" + 0.004*"public" + 0.004*"make" + 0.004*"point" + 0.003*"america" + 0.003*"state" + 0.003*"political"'),
 (4,
  '0.007*"outrage" + 0.007*"moral" + 0.005*"people" + 0.005*"good" + 0.005*"show" + 0.005*"american" + 0.005*"make" + 0.005*"president" + 0.004*"america" + 0.004*"write"')]

In [28]:
print(ldamodel.print_topics(num_topics=5, num_words=5))

# num_topics mean: how many topics want to extract 
# num_words: the number of words that want per topic

[(0, '0.008*"president" + 0.006*"moral" + 0.006*"american" + 0.006*"people" + 0.006*"outrage"'), (1, '0.014*"outrage" + 0.010*"president" + 0.005*"american" + 0.005*"make" + 0.005*"lie"'), (2, '0.010*"moral" + 0.006*"president" + 0.006*"outrage" + 0.005*"make" + 0.005*"american"'), (3, '0.006*"president" + 0.005*"american" + 0.005*"outrage" + 0.004*"moral" + 0.004*"public"'), (4, '0.007*"outrage" + 0.007*"moral" + 0.005*"people" + 0.005*"good" + 0.005*"show"')]


In [22]:
import pandas as pd
import gensim
from gensim import corpora

# Create the LDA model
ldamodel = gensim.models.ldamodel.LdaModel(doc_term_matrix, num_topics=5, 
                                           id2word=dict_, passes=20)

# Extract topics and words
topics = ldamodel.show_topics(formatted=False, num_words=20)

# Prepare data for DataFrame
data = {f'Topic {i}': [word for word, _ in topic] for i, topic in topics}

# Create DataFrame
df = pd.DataFrame(data)

# Print the DataFrame
print(df)

       Topic 0    Topic 1    Topic 2    Topic 3    Topic 4
0     american    outrage  president  president    outrage
1      outrage  president      moral   american      moral
2         make      moral   argument      write  president
3       people     people       make      moral      could
4      country        lie   american      right      write
5    president   american    outrage    america       life
6        think      death   morality    outrage    million
7        moral      point     people   morality     course
8         take      think    america    william       much
9          get       make       show        get       call
10        much     review     public      state       good
11       great     public   behavior       time       take
12       today       know        get   argument     action
13        give        say  political       make       lack
14       never      never      death         go       news
15         say       find        say       fact     benn

In [23]:
# printing the topic associations with the documents
count = 0
for i in ldamodel[doc_term_matrix]:
    print("Document",count," : ",i)
    count += 1

Document 0  :  [(2, 0.98808354)]
Document 1  :  [(1, 0.14162743), (2, 0.8533903)]
Document 2  :  [(2, 0.9964683)]
Document 3  :  [(0, 0.9786129)]
Document 4  :  [(1, 0.9874411)]
Document 5  :  [(2, 0.9796844)]
Document 6  :  [(1, 0.9688732)]
Document 7  :  [(4, 0.96595585)]
Document 8  :  [(1, 0.70806396), (3, 0.27394265)]
Document 9  :  [(4, 0.97070444)]
Document 10  :  [(1, 0.9907833)]
Document 11  :  [(0, 0.012916167), (1, 0.012739258), (2, 0.9490354), (3, 0.012700205), (4, 0.012609011)]
Document 12  :  [(1, 0.9658067)]
Document 13  :  [(3, 0.9802382)]
Document 14  :  [(3, 0.99314356)]
Document 15  :  [(0, 0.012073218), (1, 0.011935366), (2, 0.012180668), (3, 0.9518137), (4, 0.011997064)]
Document 16  :  [(4, 0.9876459)]
Document 17  :  [(0, 0.97866994)]
Document 18  :  [(3, 0.98669297)]
Document 19  :  [(1, 0.9964686)]
Document 20  :  [(0, 0.98346895)]
Document 21  :  [(0, 0.028937636), (1, 0.884206), (2, 0.028971154), (3, 0.02921716), (4, 0.028668024)]
Document 22  :  [(1, 0.98973

**=> When most of documents has high weight for topic 0, 1 so when condidering words in them we can see that they are words used for "Politics", "Biography" or "Non-fiction"  like: president/moral/people/right/american/political/reagan(name of other American politician)**

# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

In [1]:
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 
plt.style.use('ggplot')

In [2]:
df= pd.read_csv(r"/kaggle/input/amazon-books-reviews/books_data.csv")
df.head(10)

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,['David R. Ray'],http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,NaN,2000,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,['Religion'],NaN
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,['Veronica Haddon'],http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,iUniverse,2005-02,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,['Fiction'],NaN
4,"Nation Dance: Religion, Identity and Cultural ...",NaN,['Edward Long'],NaN,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,2003-03-01,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,NaN
5,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,http://books.google.nl/books?id=kVqRaiPlx88C&p...,Wm. B. Eerdmans Publishing,1996,http://books.google.nl/books?id=kVqRaiPlx88C&d...,['Religion'],5.0
6,The Overbury affair (Avon),NaN,['Miriam Allen De Ford'],NaN,http://books.google.nl/books?id=mHLTngEACAAJ&d...,NaN,1960,http://books.google.nl/books?id=mHLTngEACAAJ&d...,NaN,NaN
7,A Walk in the Woods: a Play in Two Acts,NaN,['Lee Blessing'],NaN,http://books.google.nl/books?id=6HDOwAEACAAJ&d...,NaN,1988,http://books.google.nl/books?id=6HDOwAEACAAJ&d...,NaN,3.0
8,Saint Hyacinth of Poland,The story for children 10 and up of St. Hyacin...,['Mary Fabyan Windeatt'],http://books.google.com/books/content?id=lmLqA...,http://books.google.nl/books?id=lmLqAAAACAAJ&d...,Tan Books & Pub,2009-01-01,http://books.google.nl/books?id=lmLqAAAACAAJ&d...,['Biography & Autobiography'],NaN
9,Rising Sons and Daughters: Life Among Japan's ...,Wardell recalls his experience as a foreign st...,['Steven Wardell'],NaN,http://books.google.nl/books?id=rbLZugEACAAJ&d...,Plympton PressIntl,1995,http://books.google.nl/books?id=rbLZugEACAAJ&d...,['Social Science'],NaN


In [3]:
df= pd.read_csv(r"/kaggle/input/amazon-books-reviews/books_data.csv")
df_categories=df[['Title', 'categories','description']]
df_categories

,Title,categories,description
0,Its Only Art If Its Well Hung!,['Comics & Graphic Novels'],NaN
1,Dr. Seuss: American Icon,['Biography & Autobiography'],Philip Nel takes a fascinating look into the k...
2,Wonderful Worship in Smaller Churches,['Religion'],This resource includes twelve principles in un...
3,Whispers of the Wicked Saints,['Fiction'],Julia Thomas finds her life spinning out of co...
4,"Nation Dance: Religion, Identity and Cultural ...",NaN,NaN
...,...,...,...
212399,The Orphan Of Ellis Island (Time Travel Advent...,['Juvenile Fiction'],"During a school trip to Ellis Island, Dominick..."
212400,Red Boots for Christmas,['Juvenile Fiction'],Everyone in the village of Friedensdorf is hap...
212401,Mamaw,NaN,"Give your Mamaw a useful, beautiful and though..."
212402,The Autograph Man,['Fiction'],Alex-Li Tandem sells autographs. His business ...


In [8]:
# Enable Copy-on-Write mode
pd.options.mode.copy_on_write = True

df_categories['categories'] = df_categories['categories'].astype(str)

# Filter rows where 'categories' is exactly 'Books'
books_df = df_categories[df_categories['categories'].str.contains(r'\bBooks\b', regex=True, na=False) &
                         (df_categories['categories'].str.len() <= 9)]
books_df

,Title,categories,description
2373,"Word Biblical Commentary Vol. 5, Numbers (budd...",['Books'],NaN
3567,Castle Barebane : A Novel Of Suspense,['Books'],NaN
4054,Mademoiselle liberte: Roman (French Edition),['Books'],NaN
5141,"Exile and the Kingdom (Leather Bound, Collecte...",['Books'],NaN
5353,Hijos De Dune/Children of Dune (Spanish Edition),['Books'],NaN
...,...,...,...
191667,Bread Machine Baking for All Seasons: Delightf...,['Books'],NaN
197818,Walking Away from the Third Reich (Memories Se...,['Books'],NaN
208827,The Second World War Vol. 2 - Blitzkrieg (The ...,['Books'],Stresses the role of libraries and librarians ...
210666,ngeles fugaces (Falling Angels) (Spanish Edition),['Books'],NaN


In [9]:
df= pd.read_csv(r"/kaggle/input/amazon-books-reviews/Books_rating.csv")
df

,Id,Title,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,1882931173,Its Only Art If Its Well Hung!,NaN,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",7/7,4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
1,0826414346,Dr. Seuss: American Icon,NaN,A30TK6U7DNS82R,Kevin Killian,10/10,5.0,1095724800,Really Enjoyed It,I don't care much for Dr. Seuss but after read...
2,0826414346,Dr. Seuss: American Icon,NaN,A3UH4UZ4RSVO82,John Granger,10/11,5.0,1078790400,Essential for every personal and Public Library,"If people become the books they read and if ""t..."
3,0826414346,Dr. Seuss: American Icon,NaN,A2MVUWT453QH61,"Roy E. Perry ""amateur philosopher""",7/7,4.0,1090713600,Phlip Nel gives silly Seuss a serious treatment,"Theodore Seuss Geisel (1904-1991), aka &quot;D..."
4,0826414346,Dr. Seuss: American Icon,NaN,A22X4XUPKF66MR,"D. H. Richards ""ninthwavestore""",3/3,4.0,1107993600,Good academic overview,Philip Nel - Dr. Seuss: American IconThis is b...
...,...,...,...,...,...,...,...,...,...,...
2999995,B000NSLVCU,The Idea of History,NaN,NaN,NaN,14/19,4.0,937612800,Difficult,"This is an extremely difficult book to digest,..."
2999996,B000NSLVCU,The Idea of History,NaN,A1SMUB9ASL5L9Y,jafrank,1/1,4.0,1331683200,Quite good and ahead of its time occasionally,This is pretty interesting. Collingwood seems ...
2999997,B000NSLVCU,The Idea of History,NaN,A2AQMEKZKK5EE4,"L. L. Poulos ""Muslim Mom""",0/0,4.0,1180224000,Easier reads of those not well versed in histo...,"This is a good book but very esoteric. ""What i..."
2999998,B000NSLVCU,The Idea of History,NaN,A18SQGYBKS852K,"Julia A. Klein ""knitting rat""",1/11,5.0,1163030400,"Yes, it is cheaper than the University Bookstore","My daughter, a freshman at Indiana University,..."


In [10]:
df_review=df[['User_id','Title','review/score', 'review/text']]
df_review

,User_id,Title,review/score,review/text
0,AVCGYZL8FQQTD,Its Only Art If Its Well Hung!,4.0,This is only for Julie Strain fans. It's a col...
1,A30TK6U7DNS82R,Dr. Seuss: American Icon,5.0,I don't care much for Dr. Seuss but after read...
2,A3UH4UZ4RSVO82,Dr. Seuss: American Icon,5.0,"If people become the books they read and if ""t..."
3,A2MVUWT453QH61,Dr. Seuss: American Icon,4.0,"Theodore Seuss Geisel (1904-1991), aka &quot;D..."
4,A22X4XUPKF66MR,Dr. Seuss: American Icon,4.0,Philip Nel - Dr. Seuss: American IconThis is b...
...,...,...,...,...
2999995,NaN,The Idea of History,4.0,"This is an extremely difficult book to digest,..."
2999996,A1SMUB9ASL5L9Y,The Idea of History,4.0,This is pretty interesting. Collingwood seems ...
2999997,A2AQMEKZKK5EE4,The Idea of History,4.0,"This is a good book but very esoteric. ""What i..."
2999998,A18SQGYBKS852K,The Idea of History,5.0,"My daughter, a freshman at Indiana University,..."


In [14]:
df_merge = pd.merge(books_df, df_review, on='Title', how='inner')
df_merge

,Title,categories,description,User_id,review/score,review/text
0,"Word Biblical Commentary Vol. 5, Numbers (budd...",['Books'],NaN,A3LOZIGHG4LB3X,1.0,Word Biblical Commentary: Numbers is a worst c...
1,Castle Barebane : A Novel Of Suspense,['Books'],NaN,A89TL80GBGNUK,3.0,"""Castle Barebane"" was another experiment by Jo..."
2,Castle Barebane : A Novel Of Suspense,['Books'],NaN,A3IRBCCB3I5PPD,4.0,"Joan Aiken, daughter of poet Conrad Aiken, was..."
3,Mademoiselle liberte: Roman (French Edition),['Books'],NaN,A3CM5H0P9LNO58,5.0,"Looking for a novel in Nice last week, I came ..."
4,Mademoiselle liberte: Roman (French Edition),['Books'],NaN,A1CU6NTKULREFJ,3.0,Lorsque le feu de la passion devient incontrla...
...,...,...,...,...,...,...
568,ngeles fugaces (Falling Angels) (Spanish Edition),['Books'],NaN,A1L43KWWR05PCS,4.0,"This is the Spanish text edition of the book, ..."
569,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A29DTN3W0WUBWT,3.0,When it was published in 1980 the Collazo was ...
570,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A1DA8ESF6OXRS8,5.0,"We, technical translators, urgently need a ren..."
571,Diccionario Enciclopedico de Terminos Tecnicos...,['Books'],NaN,A3G94Q5H7LWUXD,5.0,"One of the best, if not the best technical dic..."


In [15]:
from IPython.display import HTML
import pandas as pd

# Assuming df_merge is your DataFrame
df_merge.to_csv('books_new(2).csv', index=False)

# Download the file
HTML('<a href="books_new.csv" download="books_new.csv">Click here to download the CSV file</a>')